<a href="https://colab.research.google.com/github/bmulaw/ComputerVision/blob/main/ComputerVision_lab1_pixelsToFoundationModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Team Name/Member: Bernard MULAW
#### Date: 30/10/2026

In [1]:
import platform
print(platform.python_version())
# print("hello world")

3.13.18


In [2]:
%pip -q install "opencv-python" " git+https://github.com/openai/CLIP.git" "segment-anything"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
# https://pypi.org/project/segment-anything-py/, https://github.com/openai/CLIP
import clip
import torch
from PIL import Image
from segment_anything import SamPredictor, sam_model_registry, SamAutomaticMaskGenerator
import numpy as np
import cv2
import os

In [4]:
print(torch.__version__)
torch.cuda.is_available()

2.14.1+cpu


False

In [5]:
device = "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 85.4MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [6]:
class_names = ["cat", "dog", "bird", "horse"]
text = clip.tokenize([f"a photo of a {c}" for c in class_names]).to(device)

In [11]:
test_dir = "/content/drive/MyDrive/Colab" + " Notebooks/computer_vision/task1_dataset/test"

def run_clip_per_class(dir, tokneize_text):
  correct = 0
  total = 0

  per_class_accuracy = []

  for cls in class_names:
    cls_dir = os.path.join(test_dir, cls)
    new_class_accuracy_count = 0
    for img_name in sorted(os.listdir(cls_dir)):
      if not img_name.endswith(".jpg"):
        continue
      img_path = os.path.join(cls_dir, img_name)
      # clip's preprocessing (TODO: look up unsqueeze?)
      image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)

      with torch.no_grad():
        logits, _ = model(image, text)
        probs = logits.softmax(dim=-1)[0]
      pred = class_names[probs.argmax()]
      correct += (pred == cls)
      total += 1
      new_class_accuracy_count += (pred == cls)
    per_class_accuracy += [{cls: new_class_accuracy_count/25}]

  print(f"Accuracy: {correct}/{total} = {correct/total * 100:.1f}%")
  print(per_class_accuracy)

run_clip_per_class(test_dir, text)

Accuracy: 88/100 = 88.0%
[{'cat': 0.92}, {'dog': 0.88}, {'bird': 0.8}, {'horse': 0.92}]


## Task 2.1: Zero-Shot Classification

In [8]:
test_dir_10random = "/content/drive/MyDrive/Colab" + " Notebooks/computer_vision/task1_dataset/test/random10"

# TODO : Load CLIP model (ViT -B /32) --> DONE
# TODO : Define class names --> DONE
# TODO : Load all images from folder --> DONE
# TODO : For each image :
# - Preprocess image  --> DONE
# - Tokenize text --> DONE
# - Compute similarity --> DONE
# - Output : image_name -> predicted_class ( confidence ) --> DONE
def run_clip(test_dir, tokenized_text):
  correct_pred = 0
  total_predicted = 0
  for img_name in sorted(os.listdir(test_dir)):
    if not img_name.endswith(".jpg"):
      continue
    img_path = test_dir + "/" + img_name
    image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)

    with torch.no_grad():
      logits, _ = model(image, tokenized_text)
      probs = logits.softmax(dim=-1)[0]

    pred = class_names[probs.argmax()]
    confidence = round(float(max(probs)), 3)
    actual = img_name.split('_')[1]
    correct_pred += (pred == actual)
    total_predicted += 1
    print("image_class =", actual, 'model pred =', pred, "with confidence of", confidence)


  print(f"Accuracy: {correct_pred}/{total_predicted} = {correct_pred/total_predicted *100:.1f}%")

run_clip(test_dir_10random, text)


image_class = bird model pred = bird with confidence of 0.894
image_class = bird model pred = bird with confidence of 0.979
image_class = bird model pred = bird with confidence of 0.995
image_class = cat model pred = cat with confidence of 0.634
image_class = cat model pred = dog with confidence of 0.388
image_class = dog model pred = dog with confidence of 0.981
image_class = dog model pred = dog with confidence of 0.963
image_class = dog model pred = dog with confidence of 0.993
image_class = horse model pred = horse with confidence of 0.974
image_class = horse model pred = horse with confidence of 0.967
Accuracy: 9/10 = 90.0%


## Task 2.2 Prompt Engineering

In [9]:
prompt1 = clip.tokenize([f"a {c}" for c in class_names]).to(device)
prompt2 = clip.tokenize([f"a photo of a {c}" for c in class_names]).to(device)
prompt3_list = ["felis catus", "canis familiaris", "aves", "equus ferus caballus"]
prompt3 = clip.tokenize([p for p in prompt3_list]).to(device)

In [14]:
print("----------- PROMPT 1 ----------- ")
run_clip(test_dir_10random, prompt1)
print("----------- PROMPT 2 ----------- ")
run_clip(test_dir_10random, prompt2)
print("----------- PROMPT 3 ----------- ")
run_clip(test_dir_10random, prompt3)

----------- PROMPT 1 ----------- 
image_class = bird model pred = bird with confidence of 0.77
image_class = bird model pred = bird with confidence of 0.976
image_class = bird model pred = bird with confidence of 0.993
image_class = cat model pred = cat with confidence of 0.702
image_class = cat model pred = dog with confidence of 0.344
image_class = dog model pred = dog with confidence of 0.952
image_class = dog model pred = dog with confidence of 0.939
image_class = dog model pred = dog with confidence of 0.986
image_class = horse model pred = horse with confidence of 0.98
image_class = horse model pred = horse with confidence of 0.982
Accuracy: 9/10 = 90.0%
----------- PROMPT 2 ----------- 
image_class = bird model pred = bird with confidence of 0.894
image_class = bird model pred = bird with confidence of 0.979
image_class = bird model pred = bird with confidence of 0.995
image_class = cat model pred = cat with confidence of 0.634
image_class = cat model pred = dog with confidence 

## Task 2.3: Image Retrieval

In [ ]:
# TODO: Load CLIP model
# TODO: Load query + candidate images

# TODO: Compute embeddings for all images
# Hint: model.encode_image()

# TODO: Compute cosine similarity
# Similarity = embedding1 @ embedding2 .T

# TODO: Rank candidates by similarity
# Output: top -3 most similar images

# TODO: Visualize: Query image side-by-side with top-3 retrieved images

# Part 3: SAM - Interactive Segmentation

## Task 3.1: Batch Segmentation

In [ ]:
# TODO: Load SAM model
# sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth")
# sam.to(device="cuda")
# mask_generator = SamAutomaticMaskGenerator(sam)

# TODO: Loop through all images in folder

# TODO: Generate masks for each image
# masks = mask_generator.generate(image)

# TODO: Save each mask as PNG

## Task 3.2: Interactive Segmentation with Point Prompt

In [ ]:
# TODO: Load SAM model
# TODO: Load image

# TODO: Define point prompt
# input_point = np.array([[x, y]])
# input_label = np.array([1]) # 1 = foreground

# TODO: Predict mask
# mask, score, _ = predictor.predict(...)

# TODO: Visualize: image with point, mask overlay, and confidence score

# Part 4: CLIP + SAM Integration

*Build a system that takes a complext image and a text query (e.g., "the dog wearing a red collar") and gives outputs of a segmentation mask for the object described in the query and a visualization showing the mask overlaid on the image.*

- No manual annotations
- No hard-coded coordinates
- Maximum 50 lines of code
- Use CLIP + SAM only

In [ ]:
# TODO: Load CLIP

# TODO: Load SAM

# TODO: Load image and query

# TODO: Generate candidate masks

# TODO: Rank masks with CLIP

# TODO: Select best mask and visualize

In [ ]:
def clip():
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model, preprocess = clip.load("ViT -B/32", device=device)

  # Preprocess image
  image = preprocess(Image.open("image.jpg")).unsqueeze(0).to(device)

  # Tokenize text
  text = clip.tokenize(["a cat", "a dog", "a bird", "a horse"]).to(device)

  # Compute similarity
  with torch.no_grad():
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim = -1).cpu().numpy()

  # Get embedding
  with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)

In [ ]:
def sam(image, x, y):
  sam = sam_model_registry ["vit_h"](checkpoint="sam_vit_h_4b8939.pth")
  sam.to(device="cuda")

  # Automatic mask generation
  mask_generator = SamAutomaticMaskGenerator(sam)
  masks = mask_generator.generate(image)

  # Point prompt
  predictor = SamPredictor(sam)
  predictor.set_image(image)
  input_point = np.array([[x, y]])
  input_label = np.array([1])
  masks, scores, logits = predictor.predict(
    point_coords = input_point,
    point_labels = input_label,
    multimask_output = True,
  )